### Sentiment Analysis (opinion mining)

<b>Definition:</b>

Sentiment analysis is the process of determining the attitude or emotion of a writer — whether it is positive, negative, or neutral.


<b>Techniques Used:</b>

    Natural Language Processing (NLP)
    
    Text Analysis
    
    Computational Linguistics
    
    Biometrics


<b>Purpose:</b>

To systematically identify, extract, quantify, and study affective states and subjective information from text or speech.



<b> Use Cases </b>:

    Marketing – understanding customer opinions
    
    Customer Service – tracking satisfaction or complaints
    
    Clinical Medicine – analyzing patient feedback or emotional tone
    

<b> Polarity (Meaning)</b>

    Refers to the emotion expressed in a sentence.
    
    Emotions (like joy, anger, sadness) are closely tied to sentiments.
    
    The strength of a sentiment often depends on the intensity of the emotion involved.


<b>VADER  = Valence Aware Dictionary and sEntiment Reasoner </b>

It’s a lexicon and rule-based sentiment analysis tool specifically tuned for social media text (like tweets, comments, or reviews).


<b> VADER Lexicon </b> is preferred because it’s fast, rule-based, human-validated, and works exceptionally well for informal, social, and short texts — without needing training data.

In [5]:
import nltk

In [ ]:

#nltk.download('vader_lexicon')

In [6]:

from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()

VADER's SentimentIntensityAnalyzer() takes in a string and returns a dictionary of scores in each of four categories:
* negative
* neutral
* positive
* compound *(computed by normalizing the scores above)*

In [7]:

a = 'This was a good movie.'
sid.polarity_scores(a)


{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}

In [8]:

a = 'This was the best, most awesome movie EVER MADE!!!'
sid.polarity_scores(a)


{'neg': 0.0, 'neu': 0.425, 'pos': 0.575, 'compound': 0.8877}

In [9]:

a = 'This was the worst film to ever disgrace the screen.'
sid.polarity_scores(a)


{'neg': 0.477, 'neu': 0.523, 'pos': 0.0, 'compound': -0.8074}

#### Use VADER to analyze Amazon Reviews

Apply `SentimentIntensityAnalyzer` to a dataset of 10,000 Amazon reviews. 

In [10]:

import numpy as np
import pandas as pd

df = pd.read_csv('amazonreviews.tsv', sep='\t')
df

,label,review
0,pos,Stuning even for the non-gamer: This sound tra...
1,pos,The best soundtrack ever to anything.: I'm rea...
2,pos,Amazing!: This soundtrack is my favorite music...
3,pos,Excellent Soundtrack: I truly like this soundt...
4,pos,"Remember, Pull Your Jaw Off The Floor After He..."
...,...,...
9995,pos,A revelation of life in small town America in ...
9996,pos,Great biography of a very interesting journali...
9997,neg,Interesting Subject; Poor Presentation: You'd ...
9998,neg,Don't buy: The box looked used and it is obvio...


In [11]:
df['label'].value_counts()

label
neg    5097
pos    4903
Name: count, dtype: int64

#### Clean the data 


In [12]:
df.dropna(inplace=True)

In [13]:
blanks = []  

for i,lb,rv in df.itertuples():  
    if type(rv)==str:           
        if rv.isspace():        
            blanks.append(i)    

df.drop(blanks, inplace=True)

In [14]:
df['label'].value_counts()

label
neg    5097
pos    4903
Name: count, dtype: int64

#### First review through VADER

In [15]:
df.loc[0]['review']

'Stuning even for the non-gamer: This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'

In [16]:
sid.polarity_scores(df.loc[0]['review'])

{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'compound': 0.9454}

In [17]:
df.loc[0]['label']

'pos'

#### Adding Scores and Labels to the DataFrame

In [18]:
df['scores'] = df['review'].apply(lambda review: sid.polarity_scores(review))

df.head()

,label,review,scores
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co..."
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co..."
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com..."
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com..."
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp..."


In [19]:
df['compound']  = df['scores'].apply(lambda score_dict: score_dict['compound'])

df.head()

,label,review,scores,compound
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co...",0.9454
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co...",0.8957
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com...",0.9858
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com...",0.9814
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp...",0.9781


In [20]:

df['comp_score'] = df['compound'].apply(lambda c: 'pos' if c >=0 else 'neg')

df.head(10)

,label,review,scores,compound,comp_score
0,pos,Stuning even for the non-gamer: This sound tra...,"{'neg': 0.088, 'neu': 0.669, 'pos': 0.243, 'co...",0.9454,pos
1,pos,The best soundtrack ever to anything.: I'm rea...,"{'neg': 0.018, 'neu': 0.837, 'pos': 0.145, 'co...",0.8957,pos
2,pos,Amazing!: This soundtrack is my favorite music...,"{'neg': 0.04, 'neu': 0.692, 'pos': 0.268, 'com...",0.9858,pos
3,pos,Excellent Soundtrack: I truly like this soundt...,"{'neg': 0.09, 'neu': 0.615, 'pos': 0.295, 'com...",0.9814,pos
4,pos,"Remember, Pull Your Jaw Off The Floor After He...","{'neg': 0.0, 'neu': 0.746, 'pos': 0.254, 'comp...",0.9781,pos
5,pos,an absolute masterpiece: I am quite sure any o...,"{'neg': 0.014, 'neu': 0.737, 'pos': 0.249, 'co...",0.9900,pos
6,neg,"Buyer beware: This is a self-published book, a...","{'neg': 0.124, 'neu': 0.806, 'pos': 0.069, 'co...",-0.8744,neg
7,pos,Glorious story: I loved Whisper of the wicked ...,"{'neg': 0.072, 'neu': 0.583, 'pos': 0.346, 'co...",0.9900,pos
8,pos,A FIVE STAR BOOK: I just finished reading Whis...,"{'neg': 0.113, 'neu': 0.712, 'pos': 0.174, 'co...",0.8353,pos
9,pos,Whispers of the Wicked Saints: This was a easy...,"{'neg': 0.033, 'neu': 0.777, 'pos': 0.19, 'com...",0.8196,pos


In [21]:
df['comp_score'].value_counts()

comp_score
pos    6935
neg    3065
Name: count, dtype: int64

#### Report on Accuracy
Use scikit-learn to determine how close VADER came to our original 10,000 labels.

In [22]:
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [23]:
accuracy_score(df['label'],df['comp_score'])

0.7098

In [24]:
print(classification_report(df['label'],df['comp_score']))

              precision    recall  f1-score   support

         neg       0.86      0.52      0.64      5097
         pos       0.64      0.91      0.75      4903

    accuracy                           0.71     10000
   macro avg       0.75      0.71      0.70     10000
weighted avg       0.75      0.71      0.70     10000



In [25]:
print(confusion_matrix(df['label'],df['comp_score']))

[[2630 2467]
 [ 435 4468]]


### Ver 2 with TF-IDF with ML

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split


In [ ]:
X = df['review']
y = df['label']


In [ ]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),   
    stop_words='english'  
)

In [ ]:
X_tfidf = tfidf.fit_transform(X)


In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


### With Pre Process TF IDF with SVC

In [ ]:
import numpy as np
import pandas as pd
import re

In [ ]:
df = pd.read_csv("amazonreviews.tsv", sep="\t")

# Drop NaN rows
df.dropna(inplace=True)


In [ ]:
# CLEANING FUNCTION

def clean_text(t):
    t = t.lower()                                               # lowercase
    t = re.sub(r"http\S+|www\S+", "", t)                        # URLs
    t = re.sub(r"[^a-z\s]", " ", t)                             # remove punctuation & numbers
    t = re.sub(r"\b\w\b", "", t)                                # remove single characters
    t = re.sub(r"\s+", " ", t).strip()                          # normalize spaces
    return t

df['clean_review'] = df['review'].apply(clean_text)


In [ ]:


# TF-IDF VECTORIZATION

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1,2),
    stop_words='english',
    preprocessor=clean_text
)

X = tfidf.fit_transform(df['clean_review'])
y = df['label']


# TRAIN / TEST SPLIT

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
from sklearn.svm import LinearSVC

model = LinearSVC()
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))
